# DOFA + BigEarthNet Smoke Run

Colab-first template. Replace placeholder paths and repository URL before running real inference.

In [ ]:
# Clone repo
REPO_URL = "https://github.com/<your-user>/rsfm-fairness-audit.git"
!git clone {REPO_URL}
%cd rsfm-fairness-audit

In [ ]:
# Install package and base tools
!python -m pip install -e .
!python -m pip install PyYAML pytest

In [ ]:
# Install optional DOFA dependencies
!python -m pip install -r requirements-dofa.txt

In [ ]:
# Check GPU
import torch
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# Optional: mount Drive for checkpoint/data
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Configure paths. Do not hard-code private paths in commits.
PREPARED_DATA_ROOT = "/content/prepared_bigearthnet_subset"
DOFA_REPO_PATH = "/content/DOFA"
DOFA_CHECKPOINT_PATH = "/content/drive/MyDrive/<path>/DOFA_ViT_base_e100.pth"

In [ ]:
# Option A: clone official DOFA repo for local checkpoint mode
!git clone https://github.com/zhu-xlab/DOFA.git {DOFA_REPO_PATH}

In [ ]:
# Update configs/models/dofa.yaml for local repo + checkpoint mode
from pathlib import Path
import yaml

config_path = Path("configs/models/dofa.yaml")
config = yaml.safe_load(config_path.read_text())
config["repo_path"] = DOFA_REPO_PATH
config["checkpoint_path"] = DOFA_CHECKPOINT_PATH
config["allow_torch_hub_download"] = False
config["device"] = "auto"
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print(config_path.read_text())

In [ ]:
# Option B instead: enable torch.hub mode. Uncomment only if you accept checkpoint download.
# import yaml
# from pathlib import Path
# config_path = Path('configs/models/dofa.yaml')
# config = yaml.safe_load(config_path.read_text())
# config['repo_path'] = None
# config['checkpoint_path'] = None
# config['allow_torch_hub_download'] = True
# config['device'] = 'auto'
# config_path.write_text(yaml.safe_dump(config, sort_keys=False))

## Prepare Or Mount Data

Prepare `PREPARED_DATA_ROOT` so it contains `metadata.csv` and `.npy/.npz` chips. See `docs/datasets/bigearthnet_subset_setup.md`.

In [ ]:
# Example subset preparation if you already have source chips + metadata
# !python scripts/prepare_bigearthnet_subset.py \
#   --source-root <source_root> \
#   --metadata-path <source_metadata.csv> \
#   --output-root {PREPARED_DATA_ROOT} \
#   --subset-size 32 \
#   --sensor-mode S2

In [ ]:
# Run preflight checker
!python -m rsfm_fairness_audit.cli check-real \
  --model dofa \
  --dataset bigearthnet \
  --model-config configs/models/dofa.yaml \
  --data-root {PREPARED_DATA_ROOT}

In [ ]:
# Run real smoke test
!python -m rsfm_fairness_audit.cli run-real \
  --dataset bigearthnet \
  --model dofa \
  --data-root {PREPARED_DATA_ROOT} \
  --model-config configs/models/dofa.yaml \
  --subset-size 32 \
  --output-dir outputs/runs/dofa_bigearthnet_real_smoke

In [ ]:
# Inspect outputs
!find outputs/runs/dofa_bigearthnet_real_smoke -maxdepth 1 -type f -print
!sed -n '1,120p' outputs/runs/dofa_bigearthnet_real_smoke/report.md

In [ ]:
# Optional sanity run after the smoke run succeeds
# !python -m rsfm_fairness_audit.cli run-real \
#   --dataset bigearthnet \
#   --model dofa \
#   --data-root {PREPARED_DATA_ROOT} \
#   --model-config configs/models/dofa.yaml \
#   --subset-size 500 \
#   --output-dir outputs/runs/dofa_bigearthnet_real_sanity_500

In [ ]:
# Zip outputs for download
!zip -r dofa_bigearthnet_outputs.zip outputs/runs/dofa_bigearthnet_real_smoke
from google.colab import files
files.download('dofa_bigearthnet_outputs.zip')